# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1  и 1.3333 - потому что 1) none не учитывается 2) fillna заполняет пустые значениия "0" и выйдет 4/3=1.333
2. 2  - потому что "а" будет 2 
3. будет 3 потому что "df.drop_duplicates()" не записали в df
4. 1005 - отфильтрует и выдаст так
5. [9, 10] - потому что возвращает только месяц 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [55]:
import pandas as pd

df = pd.read_csv("orders.csv")
df.shape

paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
df["order_date"] = pd.to_datetime(df["order_date"])
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))

## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [10]:
# 1
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    print("прочитали:", len(df))
    
    df['amount']= pd.to_numeric(df['amount'].str.replace(" ",""))
    print("после amount:", len(df))
    
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("после city:", len(df))
    
    df = df.drop_duplicates()
    print("после дублей:", len(df))
    
    df["order_date"] = pd.to_datetime(df["order_date"])
    print("после дат:", len(df))
    
    df = df[df["customer_id"] != "TEST"]
    print("без TEST:", len(df))

    assert df["amount"].isna().sum() == 0    #assert это встроенный в python оператор проверки условий, если "False" то выдаст ошибку
    assert df["city"].isna().sum() == 0    
    assert df["order_id"].is_unique   
    
    return df   

In [11]:
df1 = clean("orders.csv")
print(df1.shape)

прочитали: 5463
после amount: 5463
после city: 5463
после дублей: 5283
после дат: 5283
без TEST: 5280
(5280, 8)


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [43]:
# 2
paid = df1["status"] == "оплачен"
sep  = df1["order_date"] < "2026-10"
octo = ~sep 
s_all = df1.loc[paid & sep, "amount"].sum()
o_all = df1.loc[paid & octo, "amount"].sum()
print(f"общая выручка: {s_all:>10,.0f} → {o_all:>10,.0f} ₸, {(o_all / s_all - 1) * 100:+.1f}%")


for city_name in df1["city"].unique():
    m = paid & (df1["city"] == city_name)
    s = df1.loc[m & sep, "amount"].sum()
    o = df1.loc[m & octo, "amount"].sum()
    print(f"{city_name:<10}: {s:>10,.0f} → {o:>10,.0f} ₸, {(o / s - 1) * 100:>+6.1f}%")

общая выручка: 35,243,410 → 34,419,520 ₸, -2.3%
Алматы    : 14,501,070 → 14,643,020 ₸,   +1.0%
Астана    :  9,034,220 →  8,850,530 ₸,   -2.0%
Караганда :  3,648,520 →  3,371,680 ₸,   -7.6%
Шымкент   :  4,558,650 →  4,044,010 ₸,  -11.3%
Атырау    :    923,990 →    969,860 ₸,   +5.0%
Актобе    :  2,576,960 →  2,540,420 ₸,   -1.4%


**Вывод по выручке.**
Выручка оплаченных заказов по чистым данным: сентябрь 35 243 410 ₸, октябрь 34 419 520 ₸, изменение −2,3%. Это совпадает с таблицей «город × месяц» с урока 4

По городам сильнее всех падает **Шымкент: −11,3%** (4 558 650 → 4 044 010 ₸). За ним Караганда (−7,6%), остальные города почти не изменились: Астана −2,0%, Актобе −1,4%, Алматы +1,0%, Атырау +5,0%

**Для менеджера:** общее падение выручки небольшое (−2,3%), а не катастрофическое. Почти всё оно приходится на Шымкент, поэтому причину нужно искать там, а не по всей компании

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [56]:
# 3
no_dups = df.drop_duplicates()                
no_test = df[~df.duplicated() & not_test]

def drop_pct(d):
    paid = d["status"] == "оплачен"
    s = d.loc[paid & (d["order_date"] < "2026-10"), "amount"].sum()
    o = d.loc[paid & (d["order_date"] >= "2026-10"), "amount"].sum()
    return (o / s - 1) * 100

print("Менеджер:   ", len(df), f"{drop_pct(df):+.1f}%")
print("Без дублей: ", len(no_dups), f"{drop_pct(no_dups):+.1f}%")
print("Без TEST:   ", len(no_test), f"{drop_pct(no_test):+.1f}%")

Менеджер:    5463 -22.7%
Без дублей:  5283 -19.6%
Без TEST:    5280 -2.3%


**Журнал очистки**

| Шаг | Строк | Убрано | Изменение выручки окт/сент |
|---|---|---|---|
| Как считал менеджер (только оплаченные) | 5 463 | 0 | −22,7% |
| Без дублей | 5 283 | 180 | −19,6% |
| Без TEST | 5 280 | 3 | −2,3% |

**Для менеджера:** сильнее всего цифру исказил клиент TEST: всего 3 тестовых заказа по 9 999 990 ₸ в сентябре дали почти 30 млн ₸ выручки, из-за чего падение выглядело как −19,6% вместо реальных −2,3%. Дубли (180 строк) тоже завышали сентябрь, но их вклад намного меньше (около 3 п.п.).

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [73]:
# 4
propusk = df1["delivery_days"].isna()
print("пропусков:", propusk.sum())
print(df1.loc[propusk, "city"].value_counts())
print(df1.loc[propusk, "status"].value_counts())

for c in df1["city"].unique():
    share = propusk[df1["city"] == c].mean()*100
    print(f"{c} {share:.1f}%")

shym = df1.loc[df1["city"] == "Шымкент", "delivery_days"]

print(f"(а) dropna:        {(shym.dropna() <= 2).mean() * 100:.1f}%")
print(f"(б) fillna(медиана): {(shym.fillna(shym.median()) <= 2).mean() * 100:.1f}%")
print(f"(в) fillna(0):     {(shym.fillna(0) <= 2).mean() * 100:.1f}%")


пропусков: 158
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
status
оплачен    144
отменён     14
Name: count, dtype: int64
Алматы 3.0%
Астана 3.0%
Караганда 4.6%
Шымкент 2.8%
Атырау 3.5%
Актобе 1.0%
(а) dropna:        29.8%
(б) fillna(медиана): 29.0%
(в) fillna(0):     31.8%


**Пропуски.** В чистых данных 158 заказов без срока доставки. Они есть во всех городах, доля везде 1–5%: ни один город не выделяется (чуть больше в Караганде, 4,6%). Среди оплаченных 144, среди отменённых 14

**Доля доставки за 2 дня в Шымкенте:** (а) dropna: 29,8%, (б) fillna(медиана): 29,0%, (в) fillna(0): 31,8%

**Честный вариант: (а) dropna.** Он считает только по заказам с известным сроком и ничего не выдумывает. fillna(0) засчитывает 22 пропуска как доставку «за 0 дней» и завышает долю. fillna(медиана) подставляет 5 дней, и пропуски записываются «медленными», поэтому доля занижена

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [74]:
# 5
df2 = pd.read_csv("orders.csv")
df2 = df2.drop_duplicates()
df2["city"] = df2["city"].str.strip().str.lower().map(CITY)
print(df2["city"].nunique(), len(df2))

6 5283


1. `df.drop_duplicates()` не записан обратно в `df`, поэтому таблица не изменилась (осталось 5 463 строки). Нужно `df = df.drop_duplicates()`
2. `str.lower()` без `strip()` оставляет написание « Алматы» с пробелом отдельным значением, а латиница (`almaty`, `astana`, `shymkent`) не совпадает с кириллицей. Поэтому 10 городов вместо 6. Нужно `.str.strip().str.lower().map(CITY)`

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [83]:
# 6
days = df1["order_date"].dt.day_name().value_counts()
print(days)

low = days.idxmin()
top = days.idxmax() 

drop = (1 - days.min() / days.max()) * 100

print(f"Меньше всего: {low} ({days.min()}), больше всего: {top} ({days.max()})")
print(f"Разница: {drop:.1f}%")

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Меньше всего: Monday (684), больше всего: Thursday (786)
Разница: 13.0%


**Заказы по дням недели.** Меньше всего заказов в понедельник (684), больше всего в четверг (786). Разница 13,0%. Воскресенье тоже слабее остальных (697), а со вторника по субботу нагрузка почти ровная (773–786)

**Для менеджера:** в понедельник и воскресенье заказов примерно на 13% меньше, чем в самые загруженные дни. Курьеров в эти дни можно ставить меньше, а основную смену планировать на вторник–субботу. Разница небольшая, поэтому сокращать график резко не нужно

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [93]:
# 7
sh = df1[df1["city"] == "Шымкент"]
other = df1[df1["city"] != "Шымкент"]

for name, d in [("Шымкент", sh), ("Остальные", other)]:
    s = d.loc[d["order_date"] < "2026-10", "delivery_days"].mean()
    o = d.loc[d["order_date"] >= "2026-10", "delivery_days"].mean()
    print(f"{name:<10}- сентябрь: {s:.2f} дн. | октябарь: {o:.2f} дн.")

last_fast =  sh.loc[sh["delivery_days"] < 5, "order_date"].max()
after = sh[sh["order_date"] > last_fast].sort_values("order_date")
first_slow = after["order_date"].iloc[0]

print("последний быстрый заказ:", last_fast)
print("первый заказ с долгой доставкой:", first_slow)
print("минимальный срок после этой даты:", after["delivery_days"].min())

for c in df1["city"].unique():
    x = df1[df1["city"] == c]
    before = x.loc[x["order_date"] < first_slow, "delivery_days"].mean()
    later = x.loc[x["order_date"] >= first_slow, "delivery_days"].mean()
    print(f"{c:<10} до: {before:.2f} | после: {later:.2f}")

Шымкент   - сентябрь: 2.03 дн. | октябарь: 6.54 дн.
Остальные - сентябрь: 2.00 дн. | октябарь: 2.01 дн.
последний быстрый заказ: 2026-09-30 22:00:00
первый заказ с долгой доставкой: 2026-10-01 10:00:00
минимальный срок после этой даты: 5.0
Алматы     до: 1.98 | после: 2.01
Астана     до: 2.04 | после: 2.00
Караганда  до: 1.90 | после: 1.99
Шымкент    до: 2.03 | после: 6.54
Атырау     до: 2.06 | после: 2.01
Актобе     до: 2.04 | после: 2.10


**Что случилось.** В Шымкенте средний срок доставки вырос с 2,0 дня в сентябре до 6,5 дня в октябре. В остальных городах срок не изменился (2,0 дня в обоих месяцах)

**Когда.** Последний заказ Шымкента с быстрой доставкой (меньше 5 дней) был 30 сентября в 22:00, а с 1 октября 2026 (первый заказ в 10:00) все заказы доставляются не быстрее чем за 5 дней 

**Что проверить в первую очередь:** что изменилось в доставке по Шымкенту ровно на стыке сентября и октября: смена службы доставки или курьерского партнёра, закрытие склада или пункта выдачи, изменение маршрутов. Это же объясняет рост отмен и падение выручки в Шымкенте, которые мы нашли раньше

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8
